# EE-2.1: Alternating current, phasors and impedance

## Objectives

At the end of this notebook you should be able to:

- describe an AC voltage by amplitude, frequency and phase, and explain what an **RMS** value is (why "230 V" peaks at 325 V)
- represent sinusoids as **phasors** — complex numbers — and add them with complex arithmetic
- derive the **impedance** of resistors, inductors and capacitors, and solve AC circuits with Ohm's law in complex form
- check a phasor result against a direct **time-domain simulation**
- solve AC networks with a complex **admittance matrix** — the Y-bus — and explain why voltage **magnitudes don't simply add** and why node voltages have **angles**

Everything in the European grid alternates 50 times per second. In DC circuits (EE-1) voltage and current were just numbers; in AC they are waves that can be shifted against each other. Phasors turn those waves back into numbers — complex ones — so that everything from EE-1 (Ohm, Kirchhoff, nodal analysis, even your `solve_fixed`) works again. This is the gateway to reactive power (EE-2.2) and AC power flow (Module 04).

In [1]:
import cmath

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

f = 50.0                      # Hz, the European grid frequency
omega = 2 * np.pi * f         # rad/s

### AC voltage and RMS values

An AC voltage is a sinusoid: **v(t) = V̂ · cos(ωt + φ)**, with peak value V̂, angular frequency ω = 2πf (f = 50 Hz in Europe, so one period lasts 20 ms) and phase φ.

The value people quote — 230 V at a socket — is not the peak. It is the **RMS** (root mean square) value: the square root of the average of v², over one period. RMS is defined so that an AC voltage delivers the **same heating power** in a resistor as a DC voltage of the same number (P = V²/R works with RMS values). For a sinusoid, RMS = V̂ / √2:

In [2]:
t = np.linspace(0, 1 / f, 200_001)[:-1]          # one period, finely sampled
v = 325.0 * np.cos(omega * t)                   # peak 325 V
v_rms = np.sqrt(np.mean(v ** 2))
print(f"RMS by numerical averaging: {v_rms:.3f} V | peak / sqrt(2): {325 / np.sqrt(2):.3f} V")

RMS by numerical averaging: 229.810 V | peak / sqrt(2): 229.810 V


The 230 V at your socket swings between +325 V and −325 V fifty times a second. From here on, **every AC magnitude is an RMS value** unless stated otherwise — the convention used in all power system tools, PyPSA and pandapower included. (The factor √2 holds for sinusoids only; the exercise shows a waveform where it fails.)

### Phasors: sinusoids as complex numbers

All voltages and currents in a linear AC circuit share the same frequency; they differ only in **magnitude** and **phase**. A **phasor** stores exactly those two numbers as one complex number:

**V = V_rms · e^{jφ}**  ↔  v(t) = √2 · V_rms · cos(ωt + φ)

(Engineers write j for √−1, because i is taken by current.) Python has complex numbers built in: `1j`, `abs(z)` for the magnitude, `np.angle(z, deg=True)` for the angle, `cmath.rect(r, phi)` to build one.

Why bother? Because **adding sinusoids of the same frequency is just adding phasors**. Check it: add two sinusoids point by point in time, and compare with the phasor sum.

In [3]:
V1 = cmath.rect(230, np.radians(0))          # 230 V at 0 degrees
V2 = cmath.rect(100, np.radians(-90))        # 100 V lagging by 90 degrees
V_sum = V1 + V2

wave = lambda P: np.sqrt(2) * abs(P) * np.cos(omega * t + np.angle(P))
assert np.allclose(wave(V1) + wave(V2), wave(V_sum)), "phasor sum must equal the time-domain sum"
print(f"phasor sum: {abs(V_sum):.2f} V at {np.angle(V_sum, deg=True):.2f} degrees")

phasor sum: 250.80 V at -23.50 degrees


The time-domain sum and the phasor sum agree at every instant. Notice that 230 V + 100 V gives **250.8 V**, not 330 V: the two waves peak at different moments. In AC, magnitudes only add when the phases are equal — keep that in mind for everything that follows.

### Impedance: resistors, inductors, capacitors

Three passive elements matter in power systems:

| Element | Time domain | Phasor domain | Impedance Z |
|---|---|---|---|
| resistor R | v = R·i | V = R·I | **R** |
| inductor L (coils, lines, transformers) | v = L·di/dt | V = jωL·I | **jωL** |
| capacitor C (cables, compensation) | i = C·dv/dt | I = jωC·V | **1/(jωC) = −j/(ωC)** |

The trick behind the middle column: differentiating e^{jωt} multiplies it by jω. So **a derivative becomes a multiplication**, and the differential equations of a circuit become algebra. Every element then obeys Ohm's law in complex form, **V = Z · I**, with impedance Z = R + jX. The imaginary part X is the **reactance**: positive for inductors, negative for capacitors, and both depend on frequency.

In [4]:
L, C = 0.05, 200e-6                            # 50 mH, 200 uF
pd.DataFrame({"impedance_ohm": [10 + 0j, 1j * omega * L, 1 / (1j * omega * C)]},
             index=["resistor 10 ohm", "inductor 50 mH", "capacitor 200 uF"]).assign(
    magnitude_ohm=lambda d: d.impedance_ohm.abs().round(3),
    angle_deg=lambda d: np.angle(d.impedance_ohm.to_numpy(), deg=True).round(1))

                         impedance_ohm  magnitude_ohm  angle_deg
resistor 10 ohm   10.000000+ 0.000000j         10.000        0.0
inductor 50 mH     0.000000+15.707963j         15.708       90.0
capacitor 200 uF   0.000000-15.915494j         15.915      -90.0

An inductor's impedance sits at +90°: its current **lags** the voltage by a quarter period. A capacitor's sits at −90°: its current **leads**. A resistor keeps current and voltage in phase.

### A series RL circuit

A 230 V source feeds a 10 Ω resistor in series with a 50 mH inductor — a crude model of a motor or of a load behind a line. In phasor form this is one division:

In [5]:
R = 10.0
Z = R + 1j * omega * L
V_src = 230 + 0j                              # reference phasor: 230 V at 0 degrees
I = V_src / Z
print(f"Z = {Z:.3f} ohm  (|Z| = {abs(Z):.3f} ohm at {np.angle(Z, deg=True):.2f} degrees)")
print(f"I = {abs(I):.3f} A at {np.angle(I, deg=True):.2f} degrees  (peak {abs(I) * np.sqrt(2):.3f} A)")

Z = 10.000+15.708j ohm  (|Z| = 18.621 ohm at 57.52 degrees)
I = 12.352 A at -57.52 degrees  (peak 17.468 A)


The current is 12.35 A RMS and lags the voltage by 57.5°. That angle is the impedance angle: the more inductive the circuit, the further the current falls behind.

### Validate: solve the differential equation directly

A phasor result is a claim about a time-domain waveform, so check it against one. The circuit obeys **L · di/dt + R · i = v(t)**. Integrate it numerically from rest, wait until the start-up transient has died away (time constant L/R = 5 ms; we wait 180 ms), then fit amplitude and phase over the last cycle:

In [6]:
v_t = lambda tt: np.sqrt(2) * 230 * np.cos(omega * tt)
sol = solve_ivp(lambda tt, i: [(v_t(tt) - R * i[0]) / L], [0, 0.2], [0.0],
                rtol=1e-10, atol=1e-12, max_step=1e-5, dense_output=True)

t_end = np.linspace(0.18, 0.2, 4001)          # last cycle: steady state
i_end = sol.sol(t_end)[0]
a, b = np.linalg.lstsq(np.column_stack([np.cos(omega * t_end), np.sin(omega * t_end)]), i_end, rcond=None)[0]
peak, phase = np.hypot(a, b), np.degrees(np.arctan2(-b, a))      # i(t) = peak * cos(wt + phase)

assert np.isclose(peak, abs(I) * np.sqrt(2), rtol=1e-6) and np.isclose(phase, np.angle(I, deg=True), atol=1e-4)
print(f"time domain: peak {peak:.6f} A at {phase:.4f} deg | phasor: {abs(I) * np.sqrt(2):.6f} A at {np.angle(I, deg=True):.4f} deg")

time domain: peak 17.467904 A at -57.5184 deg | phasor: 17.467904 A at -57.5184 deg


The numerical solution of the differential equation and the one-line phasor division agree to many digits. That is the promise of phasors: for sinusoidal steady state, the complex-number shortcut is not an approximation — it is exact. (It does *not* describe the start-up transient in the first few milliseconds; that is the territory of dynamic simulation, Module 04.)

### Kirchhoff's voltage law with phasors

KVL still holds — for **phasors**: V_R + V_L = V_source. But the **magnitudes** do not add up to the source voltage:

In [7]:
V_R, V_L = R * I, 1j * omega * L * I
assert np.isclose(V_R + V_L, V_src), "KVL holds for phasors"
print(f"|V_R| = {abs(V_R):.1f} V, |V_L| = {abs(V_L):.1f} V, sum of magnitudes {abs(V_R) + abs(V_L):.1f} V "
      f"vs source {abs(V_src):.0f} V")

|V_R| = 123.5 V, |V_L| = 194.0 V, sum of magnitudes 317.5 V vs source 230 V


123.5 V across the resistor plus 194.0 V across the inductor "add up" to 317.5 V, more than the 230 V supply — because the two voltages are 90° apart and peak at different moments. With a voltmeter you would measure exactly these magnitudes and be puzzled; with phasors, KVL closes perfectly.

### AC nodal analysis: the admittance matrix

Nodal analysis carries over directly. Replace each conductance G = 1/R by an **admittance** Y = 1/Z (a complex number), build the matrix exactly as in EE-1.1, and solve with complex arithmetic. The result is the **admittance matrix** — in power systems, the **Y-bus**, the core of every AC power flow.

The `solve_fixed` function from EE-1.2 works unchanged; only the data type becomes complex. A 230 V source feeds a load (20 + j10 Ω, resistive-inductive) through a line of 0.5 + j2 Ω — like real overhead lines, mostly reactive:

In [8]:
def admittance_matrix(branches, n_nodes):
    """Complex admittance matrix (S) including the reference node."""
    Y = np.zeros((n_nodes, n_nodes), dtype=complex)
    for a_, b_, z in branches:
        y = 1 / z
        Y[a_, a_] += y; Y[b_, b_] += y; Y[a_, b_] -= y; Y[b_, a_] -= y
    return Y


def solve_fixed_ac(branches, n_nodes, injections, fixed_V):
    """Node voltage phasors (V) and injected current phasors (A) - EE-1.2's solve_fixed, now complex."""
    Y = admittance_matrix(branches, n_nodes)
    K = sorted(fixed_V)
    U = [i for i in range(n_nodes) if i not in fixed_V]
    V = np.zeros(n_nodes, dtype=complex)
    V[K] = [fixed_V[k] for k in K]
    I_inj = np.asarray(injections, dtype=complex)
    V[U] = np.linalg.solve(Y[np.ix_(U, U)], I_inj[U] - Y[np.ix_(U, K)] @ V[K])
    return V, Y @ V


Z_line, Z_load = 0.5 + 2j, 20 + 10j
V, inj = solve_fixed_ac([(1, 2, Z_line), (2, 0, Z_load)], 3, [0, 0, 0], {0: 0, 1: 230})
I_line = (V[1] - V[2]) / Z_line
assert np.isclose(I_line, V[2] / Z_load), "KCL at node 2 with complex currents"
print(f"load voltage: {abs(V[2]):.2f} V at {np.angle(V[2], deg=True):.2f} degrees")
print(f"line current: {abs(I_line):.2f} A; |I| x |Z_line| = {abs(I_line) * abs(Z_line):.2f} V, "
      f"but |V1| - |V2| = {230 - abs(V[2]):.2f} V")

load voltage: 216.51 V at -3.78 degrees
line current: 9.68 A; |I| x |Z_line| = 19.96 V, but |V1| - |V2| = 13.49 V


Two things to notice — both central to power systems:

1. **The voltage magnitude drops less than |I|·|Z_line|**: 13.5 V instead of 20.0 V. The drop across the line is a phasor at an angle to the source voltage, so only part of it reduces the magnitude.
2. **The load voltage has an angle** (−3.8°): it is shifted in time against the source. In a real grid every bus has its own voltage angle, and — as EE-2.2 and Module 04 will show — **active power flows from buses with leading angles to buses with lagging angles**. Voltage *angles* drive active power; voltage *magnitudes* are tied to reactive power.

This is the bridge to the AC power flow equations: the Y-bus you just built, plus constant-power loads (M1.3) instead of fixed impedances, solved with Newton's method.

## Check your understanding

**AC basics**

1. A US socket delivers 120 V RMS at 60 Hz. What are its peak voltage and its period? Which impedances in the table above change at 60 Hz, and by how much?
2. Why do power engineers use RMS values rather than peak values?

**Phasors and impedance**

1. Before computing: if the inductor in the RL circuit is doubled to 100 mH, does the current grow or shrink, and does the lag increase or decrease? Check with one line of code.
2. Connect a 200 µF capacitor **in parallel** with the RL circuit. Predict what happens to the current drawn from the source — then compute it. (This is power factor correction, EE-2.2.)

**Networks**

1. In the two-node example, replace the line by a pure resistance of the same magnitude (2.06 Ω). Does the load voltage drop more or less, and what happens to its angle?
2. Which entry of the admittance matrix would you change to model a second, parallel line between nodes 1 and 2?